# Lesson 12 Lab — CUDA Execution: Grid, Block, Warp, and Divergence

**Puzzle:** A CUDA kernel launches thousands of threads; which parts are programming abstractions and which consequences are visible at warp execution?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A kernel launch defines a grid of blocks; each block contains threads with indices and can use block-scoped shared memory and barriers. Hardware schedules threads in warps under SIMT execution. Threads retain independent state, but divergent control paths within a warp may require multiple path executions with different active masks. Blocks must remain independent unless a feature explicitly provides a wider synchronization scope.


## 0. Predict before running

1. Compute the grid size for a non-multiple problem length.
2. Predict active masks for half-warp and alternating predicates.
3. Explain why launch return time is not kernel completion time.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook maps a one-dimensional problem onto grids and blocks, then evaluates three branch patterns by counting active lanes per warp and path. Uniform, half-warp, and alternating predicates can have the same true/false totals but different active-mask shapes. The metric is a transparent divergence-efficiency model, not instruction-level timing; compilers may predicate, simplify, or otherwise transform real branches.

- Grid/block/thread is the software hierarchy; warp is a hardware scheduling unit.
- A barrier must be reached by the required participating threads.
- Divergence cost depends on executed paths, active masks, and compiler behavior—not branch count alone.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["kernel launch"] --> B["grid of blocks"]
  B --> C["threads grouped into warps"]
  C --> D["predicate mask"]
  D --> E["execute path A"]
  D --> F["execute path B"]
  E --> G["reconverge"]
  F --> G
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 12
LESSON_TITLE = 'CUDA Execution: Grid, Block, Warp, and Divergence'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260825
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | uniform branch outcomes within each warp |
| Candidate | half-warp and alternating outcomes |
| Held constant | problem size, block size, warp size, and two-path assumption |
| Measurements | grid size, tail lanes, active masks, and modeled lane efficiency |
| Evidence | `numerical-model` |

**Experiment:** Map indices and compare three explicit warp branch masks.


## 6. Inspect the code

The code builds lane masks directly, calculates useful lane-work divided by issued lane slots for both paths, and reports the tail block. No custom CUDA compiler is needed to inspect the invariant.

Do not run until the code matches the frozen table.


In [2]:
problem_size = 1000
block_size = 256
warp_size = 32
grid_blocks = math.ceil(problem_size / block_size)
tail_active = problem_size - (grid_blocks - 1) * block_size

def two_path_efficiency(mask):
    true_count = sum(mask); false_count = len(mask) - true_count
    issued_paths = int(true_count > 0) + int(false_count > 0)
    return len(mask) / (len(mask) * issued_paths)

patterns = {
    "uniform": [True] * warp_size,
    "half_warp": [lane < 16 for lane in range(warp_size)],
    "alternating": [lane % 2 == 0 for lane in range(warp_size)],
}
pattern_efficiency = {name: two_path_efficiency(mask) for name, mask in patterns.items()}
metrics = {
    "problem_size": problem_size, "block_size": block_size, "warp_size": warp_size,
    "grid_blocks": grid_blocks, "tail_active_threads": tail_active,
    "patterns": pattern_efficiency,
    "active_masks_hex": {
        name: hex(sum((1 << lane) for lane, active in enumerate(mask) if active))
        for name, mask in patterns.items()
    },
}
analysis = (
    f"The launch needs {grid_blocks} blocks and the final block has {tail_active} active threads. "
    f"Under the explicit equal-cost two-path model, uniform efficiency is "
    f"{pattern_efficiency['uniform']:.1%} and both mixed patterns are "
    f"{pattern_efficiency['half_warp']:.1%}; compiler behavior is not modeled."
)
print(json.dumps(metrics, indent=2))


{
  "problem_size": 1000,
  "block_size": 256,
  "warp_size": 32,
  "grid_blocks": 4,
  "tail_active_threads": 232,
  "patterns": {
    "uniform": 1.0,
    "half_warp": 0.5,
    "alternating": 0.5
  },
  "active_masks_hex": {
    "uniform": "0xffffffff",
    "half_warp": "0xffff",
    "alternating": "0x55555555"
  }
}


## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Grid blocks | 4 |
| Tail active threads | 232 |
| Uniform efficiency | 100.00% |
| Half-warp efficiency | 50.00% |
| Alternating efficiency | 50.00% |


## 8. Explain rather than overclaim

The launch needs 4 blocks and the final block has 232 active threads. Under the explicit equal-cost two-path model, uniform efficiency is 100.0% and both mixed patterns are 50.0%; compiler behavior is not modeled.

**Evidence boundary:** A transparent mechanism model executed. It establishes the stated relationship under printed assumptions, not native hardware latency, energy, or topology.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 12, "title": 'CUDA Execution: Grid, Block, Warp, and Divergence', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Use the SIMT model to spot risky control flow, then inspect generated code and native timing before rewriting a readable branch.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 12,
  "title": "CUDA Execution: Grid, Block, Warp, and Divergence",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260825
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "problem_size": 1000,
    "block_size": 256,
    "warp_size": 32,
    "grid_blocks": 4,
    "tail_active_threads": 232,
    "patterns": {
      "uniform": 1.0,
      "half_warp": 0.5,
      "alternating": 0.5
    },
    "active_masks_hex": {
      "uniform": "0xffffffff",
      "half_warp": "0xffff",
      "alternating": "0x55555555"
    }
  },
  "analysis": "The launch needs 4 blocks and the final block has 232 active threads. Under the explicit equal-cost two-path model, uniform efficiency is 100.0% and both mixed patterns are 50.0%; compiler behavior is not modeled.",
  "conclusion": "Use the SIMT model to spot risky control flow, then inspect generat

## 10. Make the decision

> Use the SIMT model to spot risky control flow, then inspect generated code and native timing before rewriting a readable branch.

**Failure analysis:** The two-path model ignores instruction counts, reconvergence details, predication, memory divergence, and independent thread scheduling. It is not a speedup predictor.


## 11. Extend the evidence

Implement equivalent uniform and divergent CUDA kernels, inspect SASS branch/predicate instructions, and time them across path-cost ratios.

See [`README.md`](README.md) for the full explanation and references.
